# Volume 4 Drills — Policy Gradients

Work through each drill problem. Code cells are provided where applicable.
Answer conceptual questions in the markdown cells below each prompt.

## Drill 1 — Policy Gradient Theorem

**Write out the policy gradient theorem.** Fill in the blank:

$$\nabla_\theta J(\theta) = \mathbb{E}_{\pi_\theta}\left[ \underline{\hspace{4cm}} \right]$$

<details><summary>Answer</summary>

$$\nabla_\theta J(\theta) = \mathbb{E}_{\pi_\theta}\left[ \nabla_\theta \log \pi(a | s; \theta) \cdot Q^\pi(s, a) \right]$$

This says: the gradient of expected return is the expected product of the **score function** (log-policy gradient) and the **action-value** Q(s,a). We push up the log probability of actions that had high Q-values.
</details>

## Drill 2 — REINFORCE: What is G_t an Estimate of?

**Question:** In the REINFORCE algorithm, the return G_t is used as a Monte Carlo estimate. What quantity is it estimating?

**Answer (fill in):** ___

<details><summary>Answer</summary>

G_t is an unbiased (but high-variance) Monte Carlo estimate of **Q^π(s_t, a_t)** — the expected return from taking action a_t in state s_t under policy π.

$$G_t = \sum_{k=0}^{T-t} \gamma^k r_{t+k+1} \approx Q^\pi(s_t, a_t)$$
</details>

## Drill 3 — Baseline: Why No Bias?

**Question:** In policy gradients with a baseline b(s), we use `(G_t - b(s_t))` instead of `G_t`. Why does subtracting a baseline b(s) **not** bias the gradient estimate?

**Answer (fill in):** ___

<details><summary>Answer</summary>

The key identity: for any function b(s) that **does not depend on action a**:

$$\mathbb{E}_{a \sim \pi}\left[ \nabla_\theta \log \pi(a|s;\theta) \cdot b(s) \right] = b(s) \cdot \nabla_\theta \underbrace{\sum_a \pi(a|s;\theta)}_{=1} = 0$$

The gradient of a constant (w.r.t. the policy) is zero. Therefore, adding or subtracting any b(s) leaves the **expected** gradient unchanged — zero bias. However, it **reduces variance** by centering the returns.
</details>

## Drill 4 — Code: Softmax Policy in NumPy

Implement a softmax policy: given logits (unnormalized preferences), return action probabilities.

In [ ]:
import numpy as np

def softmax_policy(logits):
    """Stable softmax: subtract max for numerical stability."""
    logits = np.array(logits, dtype=float)
    logits -= logits.max()  # numerical stability
    exp_logits = np.exp(logits)
    return exp_logits / exp_logits.sum()

# Test
logits = [2.0, 1.0, 0.1]
probs = softmax_policy(logits)
print(f"Logits: {logits}")
print(f"Probs:  {probs.round(4)}")
print(f"Sum:    {probs.sum():.6f}")

# Sample an action
action = np.random.choice(len(probs), p=probs)
print(f"Sampled action: {action}")

## Drill 5 — Code: Log Probability for a Discrete Action

In [ ]:
def log_prob_discrete(logits, action):
    """Compute log π(action | logits) for a categorical distribution."""
    probs = softmax_policy(logits)
    return np.log(probs[action] + 1e-8)  # small epsilon for safety

logits = [2.0, 1.0, 0.1]
for a in range(3):
    lp = log_prob_discrete(logits, a)
    print(f"log π(a={a} | logits) = {lp:.4f}  [π(a)={softmax_policy(logits)[a]:.4f}]")

## Drill 6 — Actor-Critic: What Does the Critic Estimate?

**Question:** In an actor-critic architecture, what does the **critic** learn to estimate?

**Answer (fill in):** ___

<details><summary>Answer</summary>

The critic estimates **V^π(s)** — the expected return from state s under the current policy π. (Some variants estimate Q^π(s,a) directly.)

The critic's estimate is used as a **baseline** to reduce variance, replacing the noisy Monte Carlo return G_t with:

$$A(s, a) = Q(s, a) - V(s) \approx r + \gamma V(s') - V(s) \quad \text{(1-step TD advantage)}$$
</details>

## Drill 7 — A2C Advantage Estimate

**Question:** In A2C, the advantage A(s,a) = Q(s,a) − V(s). How is this approximated using the TD error?

Write the 1-step TD advantage formula, then compute it for: r = 1.0, γ = 0.99, V(s) = 0.5, V(s') = 0.8

<details><summary>Answer</summary>

$$A(s, a) \approx \delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$$

For the given values:

A = 1.0 + 0.99 × 0.8 − 0.5 = 1.0 + 0.792 − 0.5 = **1.292**
</details>

In [ ]:
r = 1.0
gamma = 0.99
V_s = 0.5
V_s_next = 0.8

advantage = r + gamma * V_s_next - V_s
print(f"A(s,a) = r + γ·V(s') - V(s)")
print(f"      = {r} + {gamma}×{V_s_next} - {V_s}")
print(f"      = {advantage:.4f}")

## Drill 8 — Debug: REINFORCE Using Rewards Instead of Returns

**Find the bug:**

```python
for t, (log_prob, reward) in enumerate(zip(log_probs, rewards)):
    # BUG: uses instant reward, not return G_t
    loss = -log_prob * reward
    total_loss += loss
```

**What is the correct implementation?**

<details><summary>Answer</summary>

**Bug:** REINFORCE requires the **discounted return** G_t from timestep t to end of episode, not the immediate reward r_t.

**Fix:**
```python
# Compute returns first
returns = []
G = 0
for r in reversed(rewards):
    G = r + gamma * G
    returns.insert(0, G)

# Normalize returns (optional but helpful)
returns = np.array(returns)
returns = (returns - returns.mean()) / (returns.std() + 1e-8)

for log_prob, G_t in zip(log_probs, returns):
    loss = -log_prob * G_t
    total_loss += loss
```
</details>

## Drill 9 — DDPG: Deterministic vs Stochastic Policy

**Question:** What is the key difference between a **deterministic** policy (DDPG) and a **stochastic** policy (PPO/SAC)?

**Answer (fill in):** ___

<details><summary>Answer</summary>

| | Deterministic (DDPG) | Stochastic (PPO/SAC) |
|---|---|---|
| Output | Single action: a = μ(s; θ) | Distribution: π(a\|s; θ) |
| Gradient | Deterministic policy gradient (DPG theorem) | REINFORCE / reparameterization |
| Exploration | Requires **external** noise (e.g., Ornstein-Uhlenbeck) | Built-in from sampling |
| Data | Off-policy (replay buffer) | Typically on-policy (PPO) or off-policy (SAC) |
| Action space | Continuous only | Continuous or discrete |

DDPG is simpler for continuous actions but requires careful exploration noise tuning.
</details>

## Drill 10 — TD3: The Three Tricks

**Question:** TD3 (Twin Delayed DDPG) improves on DDPG with three key tricks. Name them and explain each briefly.

**Answer (fill in):** ___

<details><summary>Answer</summary>

1. **Clipped Double Q-Learning:** Use two critic networks Q₁ and Q₂; take the **minimum** to form the TD target. Reduces overestimation bias.
   `y = r + γ · min(Q₁(s', a'), Q₂(s', a'))`

2. **Delayed Policy Updates:** Update the actor (and target networks) only every **d steps** (e.g., d=2) while updating the critics more frequently. Reduces variance in policy updates by letting critics stabilize first.

3. **Target Policy Smoothing:** Add clipped noise to target actions when computing the TD target:
   `ã = clip(μ_target(s') + clip(ε, -c, c), a_low, a_high)`, ε ~ N(0, σ)
   This acts as regularization and prevents exploiting Q-function errors at specific actions.
</details>

## Drill 11 — Code: Compute Discounted Returns

In [ ]:
def compute_returns(rewards, gamma=0.99):
    """
    Compute discounted returns G_t for each timestep t.
    G_t = r_t + γ*r_{t+1} + γ²*r_{t+2} + ...
    """
    returns = []
    G = 0.0
    for r in reversed(rewards):
        G = r + gamma * G
        returns.insert(0, G)
    return returns

# Example episode: 5 steps
rewards = [0.0, 0.0, 0.0, 0.0, 1.0]  # only final step has reward
returns = compute_returns(rewards, gamma=0.99)

for t, (r, G) in enumerate(zip(rewards, returns)):
    print(f"t={t}: r={r:.1f}  G_t={G:.6f}")

## Drill 12 — Compute: Advantage Estimate

**Compute the 1-step TD advantage** for:
- r = 1.0
- V(s') = 0.8
- V(s) = 0.5
- γ = 0.9

A = r + γ·V(s') − V(s) = ?

In [ ]:
r = 1.0
V_s_next = 0.8
V_s = 0.5
gamma = 0.9

A = r + gamma * V_s_next - V_s
print(f"A = {r} + {gamma} × {V_s_next} − {V_s}")
print(f"A = {r} + {gamma * V_s_next:.3f} − {V_s}")
print(f"A = {A:.4f}")

## Drill 13 — Why Does REINFORCE Have High Variance?

**Question:** Why does the REINFORCE algorithm typically exhibit **high variance** in gradient estimates?

**Answer (fill in):** ___

<details><summary>Answer</summary>

REINFORCE uses the **full Monte Carlo return** G_t, which:

1. **Sums many random variables** (each reward is stochastic), so variance grows with episode length.
2. **Uses a single trajectory** to estimate the expectation, which may be far from the true expected return.
3. **No bootstrapping** — every estimate starts fresh with no reuse of value estimates.

Variance is O(T) in episode length. This is why baselines (like V(s)) and actor-critic methods (which bootstrap) are critical improvements.
</details>

## Drill 14 — Code: Simple Replay Buffer for Off-Policy Methods

In [ ]:
from collections import deque
import random

class SimpleReplayBuffer:
    """Fixed-size FIFO replay buffer for off-policy RL (DDPG, TD3, SAC)."""

    def __init__(self, max_size=100_000):
        self.storage = deque(maxlen=max_size)

    def push(self, state, action, reward, next_state, done):
        self.storage.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        assert len(self) >= batch_size, "Not enough samples in buffer"
        batch = random.sample(self.storage, batch_size)
        s, a, r, ns, d = map(np.array, zip(*batch))
        return s, a, r.astype(np.float32), ns, d.astype(np.float32)

    def __len__(self):
        return len(self.storage)

# Demo
buf = SimpleReplayBuffer(max_size=1000)
for i in range(20):
    s = np.random.randn(4)
    a = np.random.randn(2)
    r = np.random.rand()
    ns = np.random.randn(4)
    d = i == 19
    buf.push(s, a, r, ns, d)

print(f"Buffer size: {len(buf)}")
states, actions, rewards, next_states, dones = buf.sample(5)
print(f"States shape: {states.shape}, Actions shape: {actions.shape}")
print(f"Rewards: {rewards}")

## Drill 15 — Challenge: Why A3C Uses Asynchronous Workers

**Question:** A3C (Asynchronous Advantage Actor-Critic) uses multiple parallel workers. Explain why asynchronous workers help training.

**Answer (fill in):** ___

<details><summary>Answer</summary>

Asynchronous workers provide two major benefits:

1. **Reduces correlation between updates:** Each worker explores a **different part of the state space** with its own copy of the environment. When gradients are asynchronously applied to the global network, consecutive updates come from different states/policies — acting like a distributed replay buffer.

2. **Faster wall-clock training:** Multiple workers collect experience **simultaneously** on CPU without needing GPU-based experience replay. A3C was shown to match or exceed DQN performance in far less wall-clock time using only CPUs.

3. **Natural curriculum:** Different workers may be at different points in training (due to lag), providing a form of implicit diversity.

Note: A2C (synchronous) is often preferred today since it's easier to reason about and implement cleanly on GPUs.
</details>